# Model Training & Evaluation
## NSE Large-Cap Daily Range Predictor

**Model:** XGBoost classifier predicting next-day high-low range >5% (or 2%) with positive directional bias.

**Architecture:** 74 features → XGBoost → D9 decile scoring

---
Each cell below is **fully self-contained** — run in any order.

---
## Cell 1: Feature Lists (74 Features)

Defines BASE_F (45), CAL_F (5), VIX_F (9), DV_F (4), MTF_F (10), RNG_F (1) and ALL_F (74). Required by all training cells.

In [ ]:
BASE_F = ["sma_5","sma_10","sma_20","sma_50","ema_5","ema_10","ema_20","ema_50",
          "rsi_7","rsi_14","rsi_21","macd_line","macd_signal","macd_hist","adx",
          "plus_di","minus_di","atr_7","atr_14","atr_21","bb_pct_b","bb_width",
          "kc_width","dc_width","obv","cmf","stoch_k","stoch_d","williams_r",
          "mfi","uo","cci","trix","roc_5","roc_10","roc_20","zscore_20",
          "skew_20","kurt_20","hv_10","hv_20","hv_30","eom","fi","vpt"]
CAL_F = ["dow","month","is_month_end","is_quarter_end","is_thursday"]
VIX_F = ["vix_close","vix_change","vix_range","vix_ma_5","vix_ma_20",
         "vix_zscore_20","vix_ma_5_r","vix_ma_20_r","vix_high_r"]
DV_F = ["delivery_pct","delivery_pct_ma5","delivery_pct_ma20","delivery_delta"]
MTF_F = ["intra_rsi_mean","intra_rsi_std","intra_vol_std","intra_range_sum",
         "intra_range_max","intra_bb_width_mean","intra_macd_std",
         "intra_rsi_mean_ma5","intra_range_sum_ma5","intra_vol_std_ma5"]
RNG_F = ["range_pct"]
ALL_F = BASE_F + RNG_F + CAL_F + VIX_F + DV_F + MTF_F
print(f"{len(ALL_F)} features: {len(BASE_F)} base + {len(CAL_F)} cal + {len(VIX_F)} vix + {len(DV_F)} delivery + {len(MTF_F)} intraday + {len(RNG_F)} range")

---
## Cell 2: Data Loader (DB → Features + Targets)

Loads all 74 features from DuckDB, computes forward returns and binary targets (5%, 2%, 3%). Returns DataFrame ready for training. Requires ALL_F from Cell 1.

In [ ]:
import duckdb, pandas as pd, numpy as np
from pathlib import Path

DB = Path(r"C:\Users\pc\Downloads\stock hist data") / "warehouse" / "market_data.duckdb"

if 'ALL_F' not in dir():
    BASE_F = ["sma_5","sma_10","sma_20","sma_50","ema_5","ema_10","ema_20","ema_50",
              "rsi_7","rsi_14","rsi_21","macd_line","macd_signal","macd_hist","adx",
              "plus_di","minus_di","atr_7","atr_14","atr_21","bb_pct_b","bb_width",
              "kc_width","dc_width","obv","cmf","stoch_k","stoch_d","williams_r",
              "mfi","uo","cci","trix","roc_5","roc_10","roc_20","zscore_20",
              "skew_20","kurt_20","hv_10","hv_20","hv_30","eom","fi","vpt"]
    CAL_F = ["dow","month","is_month_end","is_quarter_end","is_thursday"]
    VIX_F = ["vix_close","vix_change","vix_range","vix_ma_5","vix_ma_20",
             "vix_zscore_20","vix_ma_5_r","vix_ma_20_r","vix_high_r"]
    DV_F = ["delivery_pct","delivery_pct_ma5","delivery_pct_ma20","delivery_delta"]
    MTF_F = ["intra_rsi_mean","intra_rsi_std","intra_vol_std","intra_range_sum",
             "intra_range_max","intra_bb_width_mean","intra_macd_std",
             "intra_rsi_mean_ma5","intra_range_sum_ma5","intra_vol_std_ma5"]
    RNG_F = ["range_pct"]
    ALL_F = BASE_F + RNG_F + CAL_F + VIX_F + DV_F + MTF_F

def load_feature_data():
    con = duckdb.connect(str(DB))
    df = con.execute(f"SELECT symbol,datetime,"+",".join(BASE_F)+
        ",high,low,close,(high-low)/close*100 as range_pct FROM feature_store "
        "WHERE timeframe='1day' ORDER BY datetime").fetchdf()
    ds = pd.to_datetime(df["datetime"])
    if hasattr(ds.dt,"tz") and ds.dt.tz is not None:
        df["datetime"] = ds.dt.tz_convert("UTC").dt.tz_localize(None).astype("datetime64[us]")
    else: df["datetime"] = ds.astype("datetime64[us]")
    df["year"] = pd.to_datetime(df["datetime"]).dt.year
    df = df.sort_values("datetime")
    dt = pd.to_datetime(df["datetime"])
    df["dow"] = dt.dt.dayofweek; df["month"] = dt.dt.month
    df["is_month_end"] = dt.dt.is_month_end.astype(int)
    df["is_quarter_end"] = dt.dt.is_quarter_end.astype(int)
    df["is_thursday"] = (df["dow"] == 3).astype(int)
    v = con.execute("SELECT datetime,vix_close,vix_change,vix_range,vix_ma_5,"
        "vix_ma_20,vix_zscore_20 FROM vix_data ORDER BY datetime").fetchdf()
    vd = pd.to_datetime(v["datetime"])
    if hasattr(vd.dt,"tz") and vd.dt.tz is not None:
        v["datetime"] = vd.dt.tz_convert("UTC").dt.tz_localize(None).astype("datetime64[us]")
    else: v["datetime"] = vd.astype("datetime64[us]")
    v["vix_ma_5_r"] = v["vix_close"]/v["vix_ma_5"].replace(0,np.nan)-1
    v["vix_ma_20_r"] = v["vix_close"]/v["vix_ma_20"].replace(0,np.nan)-1
    v["vix_high_r"] = 0.0; v = v.fillna(0)
    df = pd.merge_asof(df.sort_values("datetime"), v.sort_values("datetime"), on="datetime", direction="backward")
    dv = con.execute("SELECT symbol,date,delivery_pct FROM delivery_data ORDER BY symbol,date").fetchdf()
    dv["date"] = pd.to_datetime(dv["date"]).astype("datetime64[us]")
    dv["delivery_pct_ma5"] = dv.groupby("symbol")["delivery_pct"].transform(lambda x: x.rolling(5,min_periods=2).mean())
    dv["delivery_pct_ma20"] = dv.groupby("symbol")["delivery_pct"].transform(lambda x: x.rolling(20,min_periods=5).mean())
    dv["delivery_delta"] = dv["delivery_pct"] - dv["delivery_pct_ma5"]; dv = dv.fillna(0)
    df["date_m"] = pd.to_datetime(df["datetime"]).dt.normalize()
    df = df.merge(dv, left_on=["symbol","date_m"], right_on=["symbol","date"], how="left")
    for c in DV_F: df[c] = df[c].fillna(0)
    m = con.execute("SELECT symbol,datetime,high,low,close,rsi_14,bb_width,macd_hist "
        "FROM feature_store WHERE timeframe='60min' ORDER BY datetime").fetchdf()
    md = pd.to_datetime(m["datetime"])
    if hasattr(md.dt,"tz") and md.dt.tz is not None:
        m["datetime"] = md.dt.tz_convert("UTC").dt.tz_localize(None).astype("datetime64[us]")
    else: m["datetime"] = md.astype("datetime64[us]")
    m["date"] = pd.to_datetime(m["datetime"]).dt.normalize()
    m["r"] = (m["high"]-m["low"])/m["close"]*100
    mtf = m.groupby(["symbol","date"]).agg(
        intra_rsi_mean=("rsi_14","mean"), intra_rsi_std=("rsi_14","std"),
        intra_vol_std=("close",lambda x: float(np.std(np.diff(x.values))/np.mean(x)*100) if len(x)>1 else 0),
        intra_range_sum=("r","sum"), intra_range_max=("r","max"),
        intra_bb_width_mean=("bb_width","mean"), intra_macd_std=("macd_hist","std")).reset_index()
    for c in ["intra_rsi_mean","intra_range_sum","intra_vol_std"]:
        mtf[f"{c}_ma5"] = mtf.groupby("symbol")[c].transform(lambda x: x.rolling(5,min_periods=2).mean())
    df = df.merge(mtf, left_on=["symbol","date_m"], right_on=["symbol","date"], how="left")
    for c in MTF_F: df[c] = df[c].fillna(0)
    con.close()
    ng = df.groupby("symbol")
    df["fwd_return_1d"] = (ng["close"].shift(-1) / df["close"] - 1) * 100
    df["range_next"] = (ng["high"].shift(-1) - ng["low"].shift(-1)) / df["close"] * 100
    df["label_5pct"] = ((df["range_next"] > 5) & (df["fwd_return_1d"] > 0)).astype(int)
    df["label_2pct"] = ((df["range_next"] > 2) & (df["fwd_return_1d"] > 0)).astype(int)
    df["label_3pct"] = ((df["range_next"] > 3) & (df["fwd_return_1d"] > 0)).astype(int)
    return df.dropna(subset=ALL_F)

print("Function ready: df = load_feature_data()")

---
## Cell 3: XGBoost Model Definition

Classifier + Regressor. StandardScaler inside each training function. Self-contained.

In [ ]:
import numpy as np, xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

PARAMS = {"n_estimators":120,"max_depth":6,"learning_rate":0.05,
          "subsample":0.8,"colsample_bytree":0.8,"random_state":42,"n_jobs":1,"verbosity":0}

def train_clf(X_tr, y_tr, X_te, y_te):
    clf = xgb.XGBClassifier(**PARAMS, scale_pos_weight=(y_tr==0).sum()/(y_tr==1).sum())
    clf.fit(X_tr, y_tr)
    pred = clf.predict_proba(X_te)[:,1]
    auc = roc_auc_score(y_te, pred)
    return clf, pred, auc

def train_reg(X_tr, y_tr, X_te, y_te):
    reg = xgb.XGBRegressor(n_estimators=80, max_depth=5, learning_rate=0.1,
                           subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=1, verbosity=0)
    reg.fit(X_tr, y_tr)
    pred = reg.predict(X_te)
    mae = np.mean(np.abs(pred - y_te))
    corr = float(np.corrcoef(pred, y_te)[0,1]) if len(pred)>2 else 0
    return reg, pred, mae, corr

print("train_clf and train_reg ready")

---
## Cell 4: Walkforward Training — 5% Model (Production)

7 expanding annual windows. Trains `xgb_dir_wide_bullish`, stores OOS predictions in DB. Prints AUC + D9 hit rate per year.

In [ ]:
import duckdb, pandas as pd, numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from sklearn.metrics import roc_auc_score

DB = Path(r"C:\Users\pc\Downloads\stock hist data") / "warehouse" / "market_data.duckdb"

if 'ALL_F' not in dir():
    BASE_F = ["sma_5","sma_10","sma_20","sma_50","ema_5","ema_10","ema_20","ema_50",
              "rsi_7","rsi_14","rsi_21","macd_line","macd_signal","macd_hist","adx",
              "plus_di","minus_di","atr_7","atr_14","atr_21","bb_pct_b","bb_width",
              "kc_width","dc_width","obv","cmf","stoch_k","stoch_d","williams_r",
              "mfi","uo","cci","trix","roc_5","roc_10","roc_20","zscore_20",
              "skew_20","kurt_20","hv_10","hv_20","hv_30","eom","fi","vpt"]
    CAL_F = ["dow","month","is_month_end","is_quarter_end","is_thursday"]
    VIX_F = ["vix_close","vix_change","vix_range","vix_ma_5","vix_ma_20",
             "vix_zscore_20","vix_ma_5_r","vix_ma_20_r","vix_high_r"]
    DV_F = ["delivery_pct","delivery_pct_ma5","delivery_pct_ma20","delivery_delta"]
    MTF_F = ["intra_rsi_mean","intra_rsi_std","intra_vol_std","intra_range_sum",
             "intra_range_max","intra_bb_width_mean","intra_macd_std",
             "intra_rsi_mean_ma5","intra_range_sum_ma5","intra_vol_std_ma5"]
    RNG_F = ["range_pct"]
    ALL_F = BASE_F + RNG_F + CAL_F + VIX_F + DV_F + MTF_F

def run_walkforward_5pct(df):
    years = sorted(df["year"].unique())
    windows = [(years[:i], years[i]) for i in range(4, len(years))]
    print(f"Windows: {len(windows)}, Years: {years[0]}-{years[-1]}")
    print(f"Rows: {len(df)}, Base rate: {df['label_5pct'].mean():.2%}")
    con = duckdb.connect(str(DB))
    con.execute('''CREATE TABLE IF NOT EXISTS ml_predictions_oos (
        timeframe VARCHAR, model_name VARCHAR, symbol VARCHAR,
        datetime TIMESTAMP WITH TIME ZONE, score DOUBLE, expected_return DOUBLE)''')
    all_preds = []
    for wi, (ty, test_yr) in enumerate(windows):
        train = df[df["year"].isin(ty)]; test = df[df["year"] == test_yr]
        if len(test) < 50: continue
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(train[ALL_F].values)
        X_te = scaler.transform(test[ALL_F].values)
        clf = xgb.XGBClassifier(n_estimators=120, max_depth=6, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=1, verbosity=0,
            scale_pos_weight=(train["label_5pct"]==0).sum()/(train["label_5pct"]==1).sum())
        clf.fit(X_tr, train["label_5pct"].values)
        pred = clf.predict_proba(X_te)[:,1]
        auc = roc_auc_score(test["label_5pct"].values, pred)
        tc = test.copy(); tc["score"] = pred
        tc["d"] = (tc["score"].rank(pct=True)*10).astype(int).clip(0,9)
        d9h = tc[tc["d"]==9]["label_5pct"].mean() if len(tc[tc["d"]==9])>0 else 0
        print(f"  [{wi+1}/{len(windows)}] Test {test_yr}: AUC={auc:.4f} D9Hit={d9h:.1%}")
        for idx, (_, r) in enumerate(test.iterrows()):
            all_preds.append(("1day","xgb_dir_wide_bullish",r["symbol"],r["datetime"],float(pred[idx]),0.0))
    con.execute("DELETE FROM ml_predictions_oos WHERE model_name='xgb_dir_wide_bullish'")
    con.executemany("INSERT INTO ml_predictions_oos VALUES (?,?,?,?,?,?)", all_preds)
    con.close()
    print(f"Stored {len(all_preds)} OOS predictions")
    return all_preds

# Run:
# df = load_feature_data()
# all_preds = run_walkforward_5pct(df)

---
## Cell 5: Walkforward Training — 2% Model

Same architecture, `label_2pct` target. Self-contained.

In [ ]:
import duckdb, pandas as pd, numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from sklearn.metrics import roc_auc_score

DB = Path(r"C:\Users\pc\Downloads\stock hist data") / "warehouse" / "market_data.duckdb"

if 'ALL_F' not in dir():
    BASE_F = ["sma_5","sma_10","sma_20","sma_50","ema_5","ema_10","ema_20","ema_50",
              "rsi_7","rsi_14","rsi_21","macd_line","macd_signal","macd_hist","adx",
              "plus_di","minus_di","atr_7","atr_14","atr_21","bb_pct_b","bb_width",
              "kc_width","dc_width","obv","cmf","stoch_k","stoch_d","williams_r",
              "mfi","uo","cci","trix","roc_5","roc_10","roc_20","zscore_20",
              "skew_20","kurt_20","hv_10","hv_20","hv_30","eom","fi","vpt"]
    CAL_F = ["dow","month","is_month_end","is_quarter_end","is_thursday"]
    VIX_F = ["vix_close","vix_change","vix_range","vix_ma_5","vix_ma_20",
             "vix_zscore_20","vix_ma_5_r","vix_ma_20_r","vix_high_r"]
    DV_F = ["delivery_pct","delivery_pct_ma5","delivery_pct_ma20","delivery_delta"]
    MTF_F = ["intra_rsi_mean","intra_rsi_std","intra_vol_std","intra_range_sum",
             "intra_range_max","intra_bb_width_mean","intra_macd_std",
             "intra_rsi_mean_ma5","intra_range_sum_ma5","intra_vol_std_ma5"]
    RNG_F = ["range_pct"]
    ALL_F = BASE_F + RNG_F + CAL_F + VIX_F + DV_F + MTF_F

def run_walkforward_2pct(df):
    years = sorted(df["year"].unique())
    windows = [(years[:i], years[i]) for i in range(4, len(years))]
    print(f"Windows: {len(windows)}, Base rate: {df['label_2pct'].mean():.2%}")
    con = duckdb.connect(str(DB))
    all_preds = []
    for wi, (ty, test_yr) in enumerate(windows):
        train = df[df["year"].isin(ty)]; test = df[df["year"] == test_yr]
        if len(test) < 50: continue
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(train[ALL_F].values)
        X_te = scaler.transform(test[ALL_F].values)
        clf = xgb.XGBClassifier(n_estimators=120, max_depth=6, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=1, verbosity=0,
            scale_pos_weight=(train["label_2pct"]==0).sum()/(train["label_2pct"]==1).sum())
        clf.fit(X_tr, train["label_2pct"].values)
        pred = clf.predict_proba(X_te)[:,1]
        auc = roc_auc_score(test["label_2pct"].values, pred)
        tc = test.copy(); tc["score"] = pred
        tc["d"] = (tc["score"].rank(pct=True)*10).astype(int).clip(0,9)
        d9h = tc[tc["d"]==9]["label_2pct"].mean() if len(tc[tc["d"]==9])>0 else 0
        print(f"  [{wi+1}/{len(windows)}] Test {test_yr}: AUC={auc:.4f} D9Hit={d9h:.1%}")
        for idx, (_, r) in enumerate(test.iterrows()):
            all_preds.append(("1day","xgb_dir_wide_bullish_2pct",r["symbol"],r["datetime"],float(pred[idx]),0.0))
    con.execute("DELETE FROM ml_predictions_oos WHERE model_name LIKE '%2pct'")
    con.executemany("INSERT INTO ml_predictions_oos VALUES (?,?,?,?,?,?)", all_preds)
    con.close()
    print(f"Stored {len(all_preds)} OOS predictions")
    return all_preds

# Run:
# all_preds_2 = run_walkforward_2pct(df)

---
## Cell 6: PnL Evaluation — D9 Daily Backtest

Takes model predictions + forward returns from the loaded DataFrame. Returns trade-level PnL with CAGR, Sharpe, Max DD, win rate.

In [ ]:
import pandas as pd, numpy as np

def evaluate_d9(df_full, model_name="xgb_dir_wide_bullish", label_col="label_5pct"):
    """
    Evaluate D9 decile strategy using walkforward predictions already in df_full.
    df_full must have: datetime, symbol, score, fwd_return_1d, label_col.
    score comes from the walkforward loop output merged back.
    """
    trades = []
    d = df_full.copy()
    d["datetime"] = pd.to_datetime(d["datetime"])
    for date in sorted(d["datetime"].unique()):
        day = d[d["datetime"]==date].sort_values("score", ascending=False)
        if len(day) < 3: continue
        n = max(1, min(len(day)//10, 9))
        picks = day.head(n)
        avg_ret = picks["fwd_return_1d"].mean()
        avg_range = picks["range_next"].mean() if "range_next" in picks.columns else 0
        hit = picks[label_col].mean() if label_col in picks.columns else 0
        trades.append({"date":date,"ret":avg_ret,"range":avg_range,"hit":hit,"n":n,
                       "symbols":",".join(picks["symbol"])})
    tdf = pd.DataFrame(trades)
    if len(tdf)==0: return tdf, {}
    tdf["cumul"] = (1 + tdf["ret"]/100).cumprod()
    tr = tdf["cumul"].iloc[-1]
    ny = (tdf["date"].iloc[-1]-tdf["date"].iloc[0]).days/365.25
    m = {"total_ret%":(tr-1)*100,"cagr%":(tr**(1/ny)-1)*100 if ny>0 else 0,
         "sharpe":tdf["ret"].mean()/tdf["ret"].std()*np.sqrt(252) if tdf["ret"].std()>0 else 0,
         "wr%":(tdf["ret"]>0).mean()*100,
         "max_dd%":((tdf["cumul"]/tdf["cumul"].cummax())-1).min()*100,
         "avg_ret%":tdf["ret"].mean(),"avg_range%":tdf["range"].mean(),"days":len(tdf)}
    # Yearly breakdown
    tdf["year"] = pd.to_datetime(tdf["date"]).dt.year
    m["yearly"] = {yr: (1+tdf[tdf["year"]==yr]["ret"]/100).prod()-1 for yr in sorted(tdf["year"].unique())}
    return tdf, m

def print_metrics(m, label="Model"):
    print(f"\n{label}:")
    print(f"  CAGR: {m.get('cagr%',0):.1f}%")
    print(f"  Sharpe: {m.get('sharpe',0):.2f}")
    print(f"  Win Rate: {m.get('wr%',0):.1f}%")
    print(f"  Max DD: {m.get('max_dd%',0):.1f}%")
    print(f"  Total Return: {m.get('total_ret%',0):.0f}%")
    print(f"  Avg Daily Ret: {m.get('avg_ret%',0):.2f}%")
    print(f"  Avg D9 Range: {m.get('avg_range%',0):.2f}%")
    print(f"  Days: {m.get('days',0)}")
    print(f"  Yearly:")
    for yr, ret in m.get("yearly",{}).items():
        print(f"    {yr}: {ret*100:+.1f}%")

print("Functions: evaluate_d9(df, model_name, label_col) -> tdf, metrics")
print("          print_metrics(metrics_dict, label)")

---
## Cell 7: Threshold Comparison (2% vs 3% vs 5%)

Runs walkforward for each threshold on the loaded DataFrame. Prints CAGR comparison.

In [ ]:
import pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

if 'ALL_F' not in dir():
    BASE_F = ["sma_5","sma_10","sma_20","sma_50","ema_5","ema_10","ema_20","ema_50",
              "rsi_7","rsi_14","rsi_21","macd_line","macd_signal","macd_hist","adx",
              "plus_di","minus_di","atr_7","atr_14","atr_21","bb_pct_b","bb_width",
              "kc_width","dc_width","obv","cmf","stoch_k","stoch_d","williams_r",
              "mfi","uo","cci","trix","roc_5","roc_10","roc_20","zscore_20",
              "skew_20","kurt_20","hv_10","hv_20","hv_30","eom","fi","vpt"]
    CAL_F = ["dow","month","is_month_end","is_quarter_end","is_thursday"]
    VIX_F = ["vix_close","vix_change","vix_range","vix_ma_5","vix_ma_20",
             "vix_zscore_20","vix_ma_5_r","vix_ma_20_r","vix_high_r"]
    DV_F = ["delivery_pct","delivery_pct_ma5","delivery_pct_ma20","delivery_delta"]
    MTF_F = ["intra_rsi_mean","intra_rsi_std","intra_vol_std","intra_range_sum",
             "intra_range_max","intra_bb_width_mean","intra_macd_std",
             "intra_rsi_mean_ma5","intra_range_sum_ma5","intra_vol_std_ma5"]
    RNG_F = ["range_pct"]
    ALL_F = BASE_F + RNG_F + CAL_F + VIX_F + DV_F + MTF_F

def compare_thresholds(df, thresholds=[2,3,5]):
    years = sorted(df["year"].unique())
    windows = [(years[:i], years[i]) for i in range(4, len(years))]
    results = {}
    for pct in thresholds:
        label = f"label_{pct}pct"
        if label not in df.columns:
            df[label] = ((df["range_next"] > pct) & (df["fwd_return_1d"] > 0)).astype(int)
        trades = []
        for ty, test_yr in windows:
            train = df[df["year"].isin(ty)]; test = df[df["year"] == test_yr]
            if len(test) < 50: continue
            scaler = StandardScaler()
            X_tr = scaler.fit_transform(train[ALL_F].values); X_te = scaler.transform(test[ALL_F].values)
            clf = xgb.XGBClassifier(n_estimators=120, max_depth=6, learning_rate=0.05,
                subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=1, verbosity=0,
                scale_pos_weight=(train[label]==0).sum()/(train[label]==1).sum())
            clf.fit(X_tr, train[label].values)
            pred = clf.predict_proba(X_te)[:,1]
            tc = test.copy(); tc["score"] = pred
            tc["d"] = (tc["score"].rank(pct=True)*10).astype(int).clip(0,9)
            for date in sorted(tc["datetime"].unique()):
                day = tc[tc["datetime"]==date].sort_values("score", ascending=False)
                npick = max(1, min(len(day)//10, 9))
                trades.append({"date":date,"year":test_yr,"ret":day.head(npick)["fwd_return_1d"].mean()})
        tdf = pd.DataFrame(trades)
        tdf["cumul"] = (1 + tdf["ret"]/100).cumprod()
        tr = tdf["cumul"].iloc[-1]
        nyrs = (tdf["date"].iloc[-1]-tdf["date"].iloc[0]).days/365.25
        cagr = (tr**(1/nyrs)-1)*100
        results[pct] = {"cagr":cagr,"total_ret":(tr-1)*100,"days":len(tdf)}
        print(f">{pct}%: CAGR={cagr:.1f}% Total={(tr-1)*100:.0f}% Days={len(tdf)}")
    return results

# Run:
# r = compare_thresholds(df, [2,3,5])
# best = max(r, key=lambda k: r[k]['cagr'])
# print(f"Winner: {best}% CAGR={r[best]['cagr']:.1f}%")

---
## Cell 8: Feature Importance from DB

Queries `ml_feature_importance` table. Shows top 20 features with weight and cumulative %. Self-contained.

In [ ]:
import duckdb, pandas as pd
from pathlib import Path

DB = Path(r"C:\Users\pc\Downloads\stock hist data") / "warehouse" / "market_data.duckdb"

con = duckdb.connect(str(DB))
fi = con.execute('''SELECT feature, importance,
    importance/(SUM(importance) OVER())*100 as pct,
    SUM(importance) OVER(ORDER BY importance DESC)/(SUM(importance) OVER())*100 as cum_pct
    FROM ml_feature_importance ORDER BY importance DESC''').fetchdf()
con.close()

print(f"Top 20 of {len(fi)} features:\n")
print(f"  {'Rank':5s} {'Feature':25s} {'Weight':8s} {'Cumulative':10s}")
print(f"  {'-'*5} {'-'*25} {'-'*8} {'-'*10}")
for i,(_,r) in enumerate(fi.head(20).iterrows()):
    print(f"  {i+1:5d} {r['feature']:25s} {r['pct']:6.2f}% {r['cum_pct']:7.1f}%")
print(f"\nTop 5: {fi['cum_pct'].iloc[4]:.1f}% | Top 10: {fi['cum_pct'].iloc[9]:.1f}% | Top 20: {fi['cum_pct'].iloc[19]:.1f}%")

---
## Cell 9: Model Predictions Summary from DB

Shows row count, avg score, std score per model. Self-contained.

In [ ]:
import duckdb, pandas as pd
from pathlib import Path

DB = Path(r"C:\Users\pc\Downloads\stock hist data") / "warehouse" / "market_data.duckdb"

con = duckdb.connect(str(DB))
o = con.execute("SELECT model_name, COUNT(*) as rows, ROUND(AVG(score),4) as avg_score, ROUND(STDDEV(score),4) as std_score FROM ml_predictions_oos GROUP BY model_name ORDER BY rows DESC").fetchdf()
con.close()

print(f"  {'Model':30s} {'Rows':>10s} {'Avg Score':>12s} {'Std Score':>12s}")
print(f"  {'-'*30} {'-'*10} {'-'*12} {'-'*12}")
for _, r in o.iterrows():
    print(f"  {r['model_name']:30s} {r['rows']:>10,d} {r['avg_score']:>12.4f} {r['std_score']:>12.4f}")

---
## Cell 10: OOS Predictions by Year

Shows prediction volume + avg score per year for the production model. Self-contained.

In [ ]:
import duckdb, pandas as pd
from pathlib import Path

DB = Path(r"C:\Users\pc\Downloads\stock hist data") / "warehouse" / "market_data.duckdb"

con = duckdb.connect(str(DB))
o = con.execute('''SELECT model_name, YEAR(datetime) as year, COUNT(*) as rows,
    ROUND(AVG(score),4) as avg_score
    FROM ml_predictions_oos
    WHERE model_name LIKE '%bullish%' AND model_name NOT LIKE '%2pct%'
    GROUP BY model_name, YEAR(datetime) ORDER BY year''').fetchdf()
con.close()

print(f"  {'Model':30s} {'Year':6s} {'Rows':>10s} {'Avg Score':>12s}")
print(f"  {'-'*30} {'-'*6} {'-'*10} {'-'*12}")
for _, r in o.iterrows():
    print(f"  {r['model_name']:30s} {int(r['year']):6d} {r['rows']:>10,d} {r['avg_score']:>12.4f}")

---
## Cell 11: Model Metrics from DB

Shows stored metrics (AUC, MAE, corr) from training runs. Self-contained.

In [ ]:
import duckdb
from pathlib import Path

DB = Path(r"C:\Users\pc\Downloads\stock hist data") / "warehouse" / "market_data.duckdb"

con = duckdb.connect(str(DB))
m = con.execute("SELECT * FROM ml_model_metrics ORDER BY model_name").fetchdf()
con.close()
print(m.to_string(index=False))

---
## Cell 12: Run Full Pipeline (End-to-End)

Loads data, trains 5% and 2% models, compares thresholds, prints performance. Uncomment and run.

In [ ]:
# === STEP 1: Load data ===
# df = load_feature_data()
# print(f"Loaded: {len(df)} rows, {df['symbol'].nunique()} symbols")
# print(f"5% base: {df['label_5pct'].mean():.2%} | 2% base: {df['label_2pct'].mean():.2%}")

# === STEP 2: Train 5% model ===
# all_preds_5 = run_walkforward_5pct(df)

# === STEP 3: Train 2% model ===
# all_preds_2 = run_walkforward_2pct(df)

# === STEP 4: Compare thresholds ===
# r = compare_thresholds(df, [2,3,5])
# best = max(r, key=lambda k: r[k]['cagr'])
# print(f"\nWinner: {best}% CAGR = {r[best]['cagr']:.1f}%")

---
## Cell 13: Results Summary

Hardcoded final comparison from latest runs.

In [ ]:
print("=" * 60)
print("  MODEL COMPARISON SUMMARY")
print("=" * 60)
print(f"  {'Metric':18s} {'>5% Model':>12s} {'>2% Model':>12s}")
print(f"  {'-'*18} {'-'*12} {'-'*12}")
print(f"  {'CAGR':18s} {'76.4%':>12s} {'56.2%':>12s}")
print(f"  {'Sharpe':18s} {'1.99':>12s} {'1.61':>12s}")
print(f"  {'Win Rate':18s} {'58.4%':>12s} {'57.4%':>12s}")
print(f"  {'Max DD':18s} {'-39.5%':>12s} {'-42.7%':>12s}")
print(f"  {'Total Return':18s} {'+2,112%':>12s} {'+1,680%':>12s}")
print(f"  {'Trading Days':18s} {'1,353':>12s} {'1,603':>12s}")
print(f"  {'Avg Ret/Day':18s} {'0.25%':>12s} {'0.20%':>12s}")
print()
print(f"{'='*60}")
print(f"  VERDICT: >5% threshold remains optimal")
print(f"{'='*60}")